In [6]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt

In [7]:
dataset_path = os.listdir("../Dataset/rooms_dataset")

In [8]:
room_type = os.listdir("../Dataset/rooms_dataset")
print(room_type)
print("types of room found: ", len(room_type))

['dining_room', 'living_room', 'bed_room']
types of room found:  3


In [9]:
rooms = []

for item in room_type:
    #get all rooms

    all_rooms = os.listdir("../Dataset/rooms_dataset" +"/"+ item)

    # adding them to list

    for img in all_rooms:
        rooms.append((item, str("../Dataset/rooms_dataset"+"/"+item) +"/"+img))
        print(rooms)

[('dining_room', '../Dataset/rooms_dataset/dining_room/render-3855578__340.jpg')]
[('dining_room', '../Dataset/rooms_dataset/dining_room/render-3855578__340.jpg'), ('dining_room', '../Dataset/rooms_dataset/dining_room/thumbnail_large-1 (1).jpg')]
[('dining_room', '../Dataset/rooms_dataset/dining_room/render-3855578__340.jpg'), ('dining_room', '../Dataset/rooms_dataset/dining_room/thumbnail_large-1 (1).jpg'), ('dining_room', '../Dataset/rooms_dataset/dining_room/interior-3674959__340.jpg')]
[('dining_room', '../Dataset/rooms_dataset/dining_room/render-3855578__340.jpg'), ('dining_room', '../Dataset/rooms_dataset/dining_room/thumbnail_large-1 (1).jpg'), ('dining_room', '../Dataset/rooms_dataset/dining_room/interior-3674959__340.jpg'), ('dining_room', '../Dataset/rooms_dataset/dining_room/lamp-689267__340.jpg')]
[('dining_room', '../Dataset/rooms_dataset/dining_room/render-3855578__340.jpg'), ('dining_room', '../Dataset/rooms_dataset/dining_room/thumbnail_large-1 (1).jpg'), ('dining_room'

In [10]:
room_df = pd.DataFrame(data = rooms, columns=['room type','image'])
room_df.head()

,room type,image
0,dining_room,../Dataset/rooms_dataset/dining_room/render-38...
1,dining_room,../Dataset/rooms_dataset/dining_room/thumbnail...
2,dining_room,../Dataset/rooms_dataset/dining_room/interior-...
3,dining_room,../Dataset/rooms_dataset/dining_room/lamp-6892...
4,dining_room,../Dataset/rooms_dataset/dining_room/villa-cor...


In [12]:
print("Total number of rooms in the dataset: ", len(room_df))

room_count = room_df['room type'].value_counts()
print(room_count)

Total number of rooms in the dataset:  118
room type
bed_room       46
living_room    38
dining_room    34
Name: count, dtype: int64


In [13]:
import cv2 as cv
path = "../Dataset/rooms_dataset/"

img_size = 224
images = []
labels = []

for i in room_type:
    data_path = path+str(i)
    filenames = [i for i in os.listdir(data_path)]
    
   # print(filenames) ## images name only
    for f in filenames:
        img = cv.imread(data_path+"/"+f)

        img = cv.resize(img, (img_size,img_size))
        images.append(img)
        labels.append(i)

labels


['dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'dining_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living_room',
 'living

In [14]:
images = np.array(images)
images.shape

(118, 224, 224, 3)

In [15]:
images = images.astype('float32')/255.0

In [16]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

y = room_df['room type'].values

y_labelEncoder = LabelEncoder()
y = y_labelEncoder.fit_transform(y)

y = y.reshape(-1,1)
onehotencoder = OneHotEncoder(sparse_output=False)
y = onehotencoder.fit_transform(y)
y.shape

(118, 3)

In [17]:
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

images , y = shuffle(images, y , random_state=1)
x_train, x_test, y_train, y_test = train_test_split(images, y, test_size=0.05, random_state=415)

print(x_train.shape, x_test.shape)
print(y_train.shape, y_test.shape)


(112, 224, 224, 3) (6, 224, 224, 3)
(112, 3) (6, 3)


## Building model

In [18]:
import tensorflow
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Input
from tensorflow.keras.preprocessing import image

In [19]:
def VGG_16(input_tensor = None, classes = 3):
    img_rows, img_cols = 224,224
    img_channels = 3
    img_dim = (img_rows, img_cols, img_channels)

    img_input = Input(shape = img_dim)

    #block 1
    x = Conv2D(64, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block1_conv1')(img_input)
    x = Conv2D(64, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block1_conv2')(x)
    x = MaxPooling2D(pool_size=(2,2), strides= (2,2), name = 'block1_pool')(x)

    #block 2
    x = Conv2D(128, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block2_conv1')(x)
    x = Conv2D(128, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block2_conv2')(x)
    x = MaxPooling2D(pool_size=(2,2), strides= (2,2), name = 'block2_pool')(x)

    #block 3
    x = Conv2D(256, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block3_conv1')(x)
    x = Conv2D(256, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block3_conv2')(x)
    x = Conv2D(256, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block3_conv3')(x)
    x = MaxPooling2D(pool_size=(2,2), strides= (2,2), name = 'block3_pool')(x)

    #block 4
    x = Conv2D(512, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block4_conv1')(x)
    x = Conv2D(512, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block4_conv2')(x)
    x = Conv2D(512, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block4_conv3')(x)
    x = MaxPooling2D(pool_size=(2,2), strides= (2,2), name = 'block4_pool')(x)

    #block 5
    x = Conv2D(512, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block5_conv1')(x)
    x = Conv2D(512, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block5_conv2')(x)
    x = Conv2D(512, kernel_size = (3,3), activation = 'relu', padding = 'same', name = 'block5_conv3')(x)
    x = MaxPooling2D(pool_size=(2,2), strides= (2,2), name = 'block5_pool')(x)

    #Flatten
    x = Flatten(name = 'flatten')(x)

    #FCL
    x = Dense(4096, activation='relu', name = 'fc1')(x)
    x = Dense(4096, activation='relu', name = 'fc2')(x)

    #output
    x = Dense(classes, activation='softmax', name = 'output')(x)

    model = Model(inputs = img_input, outputs = x, name = 'VGG16_demo')

    return model



In [20]:
model = VGG_16(classes = 3)

In [21]:
model.summary()

Model: "VGG16_demo"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 4096)           │   102,764,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 4096)           │    16,781,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 3)              │        12,291 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 134,272,835 (512.21 MB)

 Trainable params: 134,272,835 (512.21 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [23]:
history = model.fit(x_train, y_train, epochs = 4, batch_size = 32)  

Epoch 1/4
4/4 ━━━━━━━━━━━━━━━━━━━━ 112s 26s/step - accuracy: 0.2946 - loss: 16.8166
Epoch 2/4
4/4 ━━━━━━━━━━━━━━━━━━━━ 101s 24s/step - accuracy: 0.3304 - loss: 1.1072
Epoch 3/4
4/4 ━━━━━━━━━━━━━━━━━━━━ 105s 25s/step - accuracy: 0.3571 - loss: 1.0953
Epoch 4/4
4/4 ━━━━━━━━━━━━━━━━━━━━ 105s 25s/step - accuracy: 0.3750 - loss: 1.3528


In [24]:
preds = model.evaluate(x_test, y_test)
print ("Loss = " + str(preds[0]))


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.1667 - loss: 1.1191
Loss = 1.1190763711929321


In [25]:
from tensorflow.keras.applications import VGG16

In [ ]:
model = VGG16()
model.summary(). #neeed to download the model. then we can use the model

161808384/553467096 ━━━━━━━━━━━━━━━━━━━━ 4:52 1us/step

KeyboardInterrupt: 